# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: c:\Users\Lenovo\OneDrive\Bureau\Ingineria_AI
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [11]:
# Root-ul proiectului
ROOT = Path(r"c:\Users\Lenovo\OneDrive\Bureau\Ingineria_AI")

student_id = "student_3"
model = "gemini-2.5-flash-lite"
temperature = 0.2

# Fișiere
corpus_file = ROOT / "echochamber-project-team-1" / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

# Creăm folderul outputs dacă nu există
output_file.parent.mkdir(parents=True, exist_ok=True)

print("Corpus exists:", corpus_file.exists())
print("Output path:", output_file)

Corpus exists: True
Output path: c:\Users\Lenovo\OneDrive\Bureau\Ingineria_AI\outputs\student_3_prompt_outputs.jsonl


## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [12]:
records = []

with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [13]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [15]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(15)

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64

In [16]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [17]:
sample_df = df.sample(10).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
8726,RecorderRomania,"România trezește-te, oameni buni care vă doriț..."
26844,turcescu111,"Domnule Turcescu, daca considerati ca merita, ..."
22505,CălinGeorgescu-CanalulOficial,"Reporterii nu au înțeles nimic, dorm adânc din..."
14245,RecorderRomania,"A trait mult, a fost si cel mai votat, insa se..."
12596,RecorderRomania,"Sa-i aud pe unii ca-s impotriva avortului, sa ..."
19925,DianaSosoacaOfficial,"😢😢😢 In ""pandemie"" Biserica NU a făcut nimic pe..."
21291,CălinGeorgescu-CanalulOficial,UN GUVERN SI UN PRESEDINTE ILEGITIMI NU AU NIC...
6448,@CălinGeorgescu-CanalulOficial,A sosit Vremea Renașterii Romaniei. Turul 2 re...
29278,AdevaruriSecrete,Nu. Aici în momentul respectiv vor să facă com...
24874,turcescu111,Mergeți din curiozitate la Casa de pensii Sect...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [18]:
SYSTEM_PROMPT = """
Ești un analist de discurs politic online. 
Analizezi comentarii politice românești și extragi informații descriptive despre poziționare, ton și interpretare.

Răspunsurile trebuie să fie concise, clare și să respecte strict formatul JSON cerut.
Nu inventa informații care nu apar în comentariu.
"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu politic și identifică:

1. target:
persoana, partidul, instituția sau grupul principal vizat de comentariu.

2. stance:
poziționarea autorului față de țintă:
- pro
- anti
- neutru
- mixt

3. sentiment:
sentimentul general al comentariului:
- pozitiv
- negativ
- neutru
- mixt

4. tone:
stilul discursiv dominant:
exemple: ironic, furios, agresiv, admirativ, sarcastic, conspirativ, defensiv, alarmist, moderat etc.

5. topic:
tema principală discutată:
exemple: alegeri, corupție, Rusia, UE, propagandă, economie, democrație etc.

6. interpretation_problem:
spune dacă există o problemă de interpretare:
- ambiguitate
- sarcasm
- conspirație
- lipsă context
- niciuna

7. justification:
o explicație foarte scurtă (1-2 propoziții) pentru clasificare.

Important:
- stance NU este același lucru cu sentimentul.
- Un comentariu poate avea ton negativ dar stance pro față de țintă.
- Folosește doar informația din comentariu.
- Returnează DOAR JSON valid.

Returnează JSON valid cu exact aceste chei:
target, stance, sentiment, tone, topic, interpretation_problem, justification

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [29]:
from openai import OpenAI
client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [30]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [31]:
n_comments = 10  # schimbă aici: 3, 5 sau 10
sample_for_prompt = df.sample(n_comments).copy()

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,RecorderRomania,PORTRET DE CANDIDAT: Nicușor Dan,De departe varianta cea mai sigura este Nicuso...,"```json\n{\n ""target"": ""Nicosor Dan"",\n ""sta..."
1,georgesimionoficial,Românii din diaspora au scris istorie: votul l...,Mai facut sa pling stimate george simion astep...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
2,turcescu111,"Coșmarul nu poate dura cinci ani, au înțeles a...","ZILE FRUMOASE SĂ AVEȚI CU FAMILIA,AVETI DREPTA...","```json\n{\n ""target"": ""Mucușor"",\n ""stance""..."
3,CălinGeorgescu-CanalulOficial,Călin Georgescu - Zidurile minciunii vor căde...,Romanii sunt in pragul revoltei. Au venit fact...,"```json\n{\n ""target"": ""Guvernul/Autoritățile..."
4,turcescu111,Georgescu le-a dat la operație!,NICI NU AM RESPIRAT CÂT TIMP A FOST PREȘEDINTE...,"```json\n{\n ""target"": ""Călin Georgescu și Io..."
5,RecorderRomania,Investigație cu camera ascunsă. Cât de ușor aj...,"Multe permise, putini soferi. Un sofer este ce...","```json\n{\n ""target"": ""soferi"",\n ""stance"":..."
6,turcescu111,"Avem buget, dar n-avem bani. Războiul e tatăl ...",Cred ca marea problema a SuA e Israel! Toate r...,"```json\n{\n ""target"": ""Statele Unite ale Ame..."
7,RecorderRomania,Investigație cu camera ascunsă. Cât de ușor aj...,am permisul de 3 ani de zile și încă îmi este ...,"```json\n{\n ""target"": ""școala de șoferi"",\n ..."
8,georgesimionoficial,"Azi la Răcășdia, printre români","George Simion, președinte! Reîntoarcerea la de...","```json\n{\n ""target"": ""George Simion"",\n ""s..."
9,RecorderRomania,DOCUMENTAR RECORDER. Justiție capturată,Multumesc pentru curaj si devotament! Informat...,"```json\n{\n ""target"": ""Autorul comentariului..."


# 9. Verificam rezultatele

In [32]:
results_df.model_output[0]

'```json\n{\n  "target": "Nicosor Dan",\n  "stance": "pro",\n  "sentiment": "pozitiv",\n  "tone": "admirativ",\n  "topic": "alegeri",\n  "interpretation_problem": "niciuna",\n  "justification": "Autorul consideră Nicușor Dan \\"cea mai sigură variantă\\" și își exprimă speranța ca alegătorii să ia o decizie \\"corectă\\", indicând o poziționare pro și un sentiment pozitiv față de acesta."\n}\n```'

In [33]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [34]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,RecorderRomania,PORTRET DE CANDIDAT: Nicușor Dan,De departe varianta cea mai sigura este Nicuso...,Nicosor Dan,pro,pozitiv,admirativ,alegeri,niciuna,
1,georgesimionoficial,Românii din diaspora au scris istorie: votul l...,Mai facut sa pling stimate george simion astep...,George Simion,pro,pozitiv,admirativ,politică internă,niciuna,
2,turcescu111,"Coșmarul nu poate dura cinci ani, au înțeles a...","ZILE FRUMOASE SĂ AVEȚI CU FAMILIA,AVETI DREPTA...",Mucușor,pro,pozitiv,admirativ,persoane,niciuna,
3,CălinGeorgescu-CanalulOficial,Călin Georgescu - Zidurile minciunii vor căde...,Romanii sunt in pragul revoltei. Au venit fact...,Guvernul/Autoritățile,anti,negativ,alarmist,economie/costul vieții,niciuna,
4,turcescu111,Georgescu le-a dat la operație!,NICI NU AM RESPIRAT CÂT TIMP A FOST PREȘEDINTE...,"Călin Georgescu și Ion Cristoiu (presupus, pri...",pro,pozitiv,admirativ,politică,niciuna,
5,RecorderRomania,Investigație cu camera ascunsă. Cât de ușor aj...,"Multe permise, putini soferi. Un sofer este ce...",soferi,neutru,neutru,explicativ,soferie,niciuna,
6,turcescu111,"Avem buget, dar n-avem bani. Războiul e tatăl ...",Cred ca marea problema a SuA e Israel! Toate r...,Statele Unite ale Americii (SUA) și relația lo...,anti,negativ,conspirativ,"politică externă, relații internaționale",conspirație,
7,RecorderRomania,Investigație cu camera ascunsă. Cât de ușor aj...,am permisul de 3 ani de zile și încă îmi este ...,școala de șoferi,anti,negativ,furios,educație,niciuna,
8,georgesimionoficial,"Azi la Răcășdia, printre români","George Simion, președinte! Reîntoarcerea la de...",George Simion,pro,pozitiv,admirativ,politică,niciuna,
9,RecorderRomania,DOCUMENTAR RECORDER. Justiție capturată,Multumesc pentru curaj si devotament! Informat...,Autorul comentariului (implicit),pro,pozitiv,admirativ,informare,niciuna,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [35]:
# salvare CSV
results_df.to_csv(output_file.with_suffix(".csv"), index=False, encoding="utf-8-sig")

print("Fisier salvat:", output_file.with_suffix(".csv"))

Fisier salvat: c:\Users\Lenovo\OneDrive\Bureau\Ingineria_AI\outputs\student_3_prompt_outputs.csv
